In [1]:
import queue
from typing import Dict, List

class RabbitMQExchange:
    def __init__(self, name: str):
        self.name = name
        # Routing_key -> Kuyruk Listesi eşleşmesi
        self.bindings: Dict[str, List['RabbitMQQueue']] = {}

    def bind(self, routing_key: str, q: 'RabbitMQQueue'):
        if routing_key not in self.bindings:
            self.bindings[routing_key] = []
        self.bindings[routing_key].append(q)

    def publish(self, routing_key: str, message: dict):
        if routing_key in self.bindings:
            for q in self.bindings[routing_key]:
                q.enqueue(message)
                print(f"[RabbitMQ - {self.name}] '{routing_key}' ile '{q.name}' kuyruğuna iletildi: {message}")
        else:
            print(f"[RabbitMQ] Eşleşen kuyruk bulunamadı: {routing_key}")

class RabbitMQQueue:
    def __init__(self, name: str):
        self.name = name
        self._queue = queue.Queue() # Thread-safe FIFO kuyruk

    def enqueue(self, message: dict):
        self._queue.put(message)

    def consume(self):
        """Mesaj çekilir (Broker tüketiciye teslim eder)"""
        if not self._queue.empty():
            return self._queue.get()
        return None

    def ack(self):
        """İşlem tamamlandı, mesaj broker'dan tamamen silindi"""
        self._queue.task_done()
        print(f"[RabbitMQ - {self.name}] Mesaj onaylandı (ACK) ve kuyruktan SİLİNDİ.")

In [2]:
from typing import List, Dict

class KafkaPartition:
    def __init__(self, partition_id: int):
        self.partition_id = partition_id
        # Append-only Log (Mesajlar asla doğrudan silinmez)
        self.log: List[dict] = []

    def append(self, message: dict) -> int:
        self.log.append(message)
        offset = len(self.log) - 1
        return offset

    def read_from(self, offset: int):
        if offset < len(self.log):
            return self.log[offset]
        return None

class KafkaTopic:
    def __init__(self, name: str, num_partitions: int = 2):
        self.name = name
        self.partitions = [KafkaPartition(i) for i in range(num_partitions)]

    def produce(self, key: str, message: dict):
        # Basit bir hash ile Partition seçimi (Kafka'nın gerçek mantığı)
        partition_id = hash(key) % len(self.partitions)
        offset = self.partitions[partition_id].append(message)
        print(f"[Kafka - {self.name}] Partition:{partition_id}, Offset:{offset}'e yazıldı: {message}")

class KafkaConsumer:
    def __init__(self, topic: KafkaTopic, partition_id: int):
        self.topic = topic
        self.partition_id = partition_id
        self.offset = 0 # Tüketicinin kendi takip ettiği imleç

    def poll(self):
        """Kafka'da tüketici 'Pull' yapar, mesaj silinmez"""
        partition = self.topic.partitions[self.partition_id]
        msg = partition.read_from(self.offset)
        if msg:
            print(f"[Kafka Consumer] Partition:{self.partition_id}, Offset:{self.offset} okundu: {msg}")
            self.offset += 1 # Bir sonraki mesaja geç
            return msg
        return None

    def seek(self, new_offset: int):
        """Kafka mucizesi: Zamanda geriye gitme (Replay) imkanı!"""
        self.offset = new_offset
        print(f"[Kafka Consumer] Offset {new_offset} konumuna geri sarıldı.")

In [3]:
if __name__ == "__main__":
    print("=== 1. RABBITMQ SİMÜLASYONU ===")
    exchange = RabbitMQExchange("order_exchange")
    order_queue = RabbitMQQueue("orders_queue")
    exchange.bind("order.created", order_queue)

    # 1. Mesajı yolla
    exchange.publish("order.created", {"order_id": 101, "amount": 250})

    # 2. Mesajı tüket ve ACK gönder
    msg = order_queue.consume()
    print(f"İşleniyor: {msg}")
    order_queue.ack()

    # 3. Tekrar okumayı dene (Kuyruk boş çünkü silindi!)
    print("Tekrar okuma deneniyor:", order_queue.consume())


    print("\n" + "="*40 + "\n")


    print("=== 2. KAFKA SİMÜLASYONU ===")
    orders_topic = KafkaTopic("orders_stream", num_partitions=1)
    consumer = KafkaConsumer(orders_topic, partition_id=0)

    # 1. Log'a 2 mesaj ekle
    orders_topic.produce(key="user_1", message={"order_id": 201, "amount": 100})
    orders_topic.produce(key="user_1", message={"order_id": 202, "amount": 500})

    # 2. Tüketici mesajları sırayla çeker (Poll)
    consumer.poll()
    consumer.poll()

    # 3. Kafka Farkı: Mesajlar silinmedi! Başa sarıp (Replay) tekrar okuyabiliriz:
    print("\n-- Kafka Offset Başa Alınıyor (Replay) --")
    consumer.seek(0)
    consumer.poll() # 201 nolu mesajı tekrar okur!

=== 1. RABBITMQ SİMÜLASYONU ===
[RabbitMQ - order_exchange] 'order.created' ile 'orders_queue' kuyruğuna iletildi: {'order_id': 101, 'amount': 250}
İşleniyor: {'order_id': 101, 'amount': 250}
[RabbitMQ - orders_queue] Mesaj onaylandı (ACK) ve kuyruktan SİLİNDİ.
Tekrar okuma deneniyor: None


=== 2. KAFKA SİMÜLASYONU ===
[Kafka - orders_stream] Partition:0, Offset:0'e yazıldı: {'order_id': 201, 'amount': 100}
[Kafka - orders_stream] Partition:0, Offset:1'e yazıldı: {'order_id': 202, 'amount': 500}
[Kafka Consumer] Partition:0, Offset:0 okundu: {'order_id': 201, 'amount': 100}
[Kafka Consumer] Partition:0, Offset:1 okundu: {'order_id': 202, 'amount': 500}

-- Kafka Offset Başa Alınıyor (Replay) --
[Kafka Consumer] Offset 0 konumuna geri sarıldı.
[Kafka Consumer] Partition:0, Offset:0 okundu: {'order_id': 201, 'amount': 100}
